In [1]:
""" default calculation for Gaines-Thomas exchange coefficients are probably not correct for UIEF """

from typing import Dict

def exchange_coeffs(sand: float, clay: float, om: float, zsoi: float) -> Dict[str, float]:
    """
    Compute log K (exchange) coefficients for Ca2+, Mg2+, Na+, K+, Al3+.

    Parameters
    ----------
    sand : float
        Sand percentage (0–100).
    clay : float
        Clay percentage (0–100).
    om : float
        Organic matter as fraction (0–1). If >1, treated as percent and divided by 100.
    zsoi : float
        Soil depth in meters.

    Returns
    -------
    Dict[str, float]
        Dictionary with keys 'Ca2+', 'Mg2+', 'Na+', 'K+', 'Al3+' and corresponding log K values.
    """
    # Convert OM to fraction if given in percent
    om_frac = om / 100.0 if om > 1.0 else om

    # Choose kex coefficients by depth (exactly as in the Fortran)
    if zsoi <= 0.1:
        kex_ca = (3.194, 3.238, 3.728, 2.778)
        kex_mg = (3.552, 3.567, 4.095, 3.134)
        kex_k  = (1.936, 1.612, 2.180, 1.828)
        kex_na = (2.713, 3.085, 3.603, 2.512)
        kex_al = (2.872, 2.948, 3.996, 3.191)
    elif zsoi <= 0.3:
        kex_ca = (3.593, 3.424, 3.333, 2.724)
        kex_mg = (3.824, 3.688, 3.678, 3.134)
        kex_k  = (2.245, 1.675, 1.687, 1.911)
        kex_na = (3.320, 2.620, 3.152, 2.504)
        kex_al = (2.834, 2.858, 3.750, 3.183)
    elif zsoi <= 0.6:
        kex_ca = (3.769, 3.6625, 3.556, 2.947)
        kex_mg = (3.977, 3.9225, 3.868, 3.367)
        kex_k  = (2.227, 2.1975, 2.168, 2.263)
        kex_na = (3.223, 3.466 , 3.709, 2.866)
        kex_al = (2.843, 3.92  , 4.997, 3.683)
    else:
        kex_ca = (4.199, 3.404, 3.640, 3.069)
        kex_mg = (4.342, 3.559, 4.241, 3.354)
        kex_k  = (1.714, 1.924, 2.518, 2.318)
        kex_na = (2.959, 2.230, 4.077, 2.825)
        kex_al = (2.752, 2.841, 5.471, 3.894)

    # Helper to compute a single cation's value (exact formula from Fortran)
    def calc(kex):
        # Note the ordering: kex(1)*sand + kex(3)*clay + kex(2)*silt
        silt = 100.0 - sand - clay
        term_mineral = (kex[0] * sand + kex[2] * clay + kex[1] * silt) * (1.0 - om_frac) / 100.0
        term_om = kex[3] * om_frac
        return -(term_mineral + term_om)

    return {
        "Ca2+": calc(kex_ca),
        "Mg2+": calc(kex_mg),
        "Na+":  calc(kex_na),
        "K+":   calc(kex_k),
        "Al3+": calc(kex_al),
    }

# Example:
results = exchange_coeffs(sand=18.0, clay=22.0, om=3.0/130, zsoi=0.2)
print(results)

{'Ca2+': -3.4180061538461537, 'Mg2+': -3.6969812307692305, 'Na+': -2.854754461538462, 'K+': -1.7832575384615386, 'Al3+': -3.0529910769230773}


In [ ]:
# Create an exchange coefficient table

# Table 3 in:
# Jalali, M., Arian, T. M., & Ranjbar, F. (2020). Selectivity coefficients of K, Na, Ca, and Mg in 
# binary exchange systems in some calcareous soils. Environmental Monitoring and Assessment, 192(2),
# 80. https://doi.org/10.1007/s10661-019-8022-y
#   "So, the affinity sequence of cations for the exchange sites of soils in this study can be
#    arranged as: K > Ca > Mg > Na."

import pandas as pd
import numpy as np

table_Jalali = pd.DataFrame(np.nan, index = ['Ca','Mg','K','Na'], columns = ['Ca','Mg','K','Na'])

table_Jalali.loc['K','Ca'] = 16.5
table_Jalali.loc['K','Mg'] = 7.8
table_Jalali.loc['K', 'Na'] = 1.7
table_Jalali.loc['Na', 'Ca'] = 0.5
table_Jalali.loc['Na', 'Mg'] = 0.1
table_Jalali.loc['Na', 'K'] = 0.1
table_Jalali.loc['Ca','Mg'] = 4.1
table_Jalali.loc['Ca', 'Na'] = 0.8
table_Jalali.loc['Ca', 'K'] = 0.6   # overwrites the previous Ca-Na entry
table_Jalali.loc['Mg', 'Ca'] = 3.1
table_Jalali.loc['Mg', 'Na'] = 2.1
table_Jalali.loc['Mg', 'K'] = 1.0

print(table_Jalali)

      Ca   Mg    K   Na
Ca   NaN  4.1  0.6  0.8
Mg   3.1  NaN  1.0  2.1
K   16.5  7.8  NaN  1.7
Na   0.5  0.1  0.1  NaN
